#1 Basic Installation

In [56]:
import pandas as pd
import numpy as np
import os
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#2 Loading DataSets

In [57]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("jquigl/imdb-genres")

train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])

print("Train size:", train_df.shape)
print("Validation size:", val_df.shape)

train_df.head()

Train size: (238256, 5)
Validation size: (29809, 5)


,movie title - year,genre,expanded-genres,rating,description
0,Flaming Ears - 1992,Fantasy,"Fantasy, Sci-Fi",6.0,Flaming Ears is a pop sci-fi lesbian fantasy f...
1,Jeg elsker dig - 1957,Romance,"Comedy, Drama, Romance",5.8,Six people - three couples - meet at random at...
2,Povjerenje - 2021,Thriller,Thriller,NaN,"In a small unnamed town, in year 2025, Krsto w..."
3,Gulliver Returns - 2021,Fantasy,"Animation, Adventure, Family",4.4,The legendary Gulliver returns to the Kingdom ...
4,Prithvi Vallabh - 1924,Biography,"Biography, Drama, Romance",NaN,"Seminal silent historical film, the story feat..."


In [58]:
train_df = train_df.rename(columns={
    "movie title - year": "title",
    "rating": "ratings"
})

val_df = val_df.rename(columns={
    "movie title - year": "title",
    "rating": "ratings"
})

print(train_df.columns)

Index(['title', 'genre', 'expanded-genres', 'ratings', 'description'], dtype='object')


In [59]:
train_df = train_df[['title','description','ratings','genre']]
val_df = val_df[['title','description','ratings','genre']]

train_df = train_df.dropna()
val_df = val_df.dropna()

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

#3 Text Preprocessing

In [60]:
import string
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [61]:
train_df["clean_description"] = train_df["description"].apply(clean_text)
val_df["clean_description"] = val_df["description"].apply(clean_text)

#4 Model Training by using TF-IDF

In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
train_tfidf = tfidf.fit_transform(train_df["clean_description"])
val_tfidf = tfidf.transform(val_df["clean_description"])

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(title=None, description=None, genre=None, top_n=5):

    if title:
        title = title.strip().lower()

        #For  Finding closest match
        matches = train_df[train_df['title'].str.lower().str.contains(title)]

        if matches.empty:
            return "Movie not found"

        idx = matches.index[0]

        query_vector = train_tfidf[idx]
        sim_scores = cosine_similarity(query_vector, train_tfidf)[0]

    elif description:
        description = clean_text(description)
        query_vector = tfidf.transform([description])
        sim_scores = cosine_similarity(query_vector, train_tfidf)[0]

    else:
        return "Provide title or description"

    sim_indices = sim_scores.argsort()[::-1][1:top_n+20]
    results = train_df.iloc[sim_indices]

    if genre:
        results = results[results['genre'].str.contains(genre, case=False, na=False)]

    return results[['title', 'genre', 'ratings']].head(top_n)

In [64]:
indices = pd.Series(train_df.index, index=train_df['title']).drop_duplicates()

In [65]:

print(recommend_movies(title="The Dark Knight"))

                                   title      genre  ratings
143191  Kitaro's Graveyard Gang 2 - 2011  Animation      7.3
129572           Batman and Robin - 1949     Family      5.9
150543           Batman and Robin - 1949      Scifi      5.9
13101            Batman and Robin - 1949     Action      5.9
81537            Batman and Robin - 1949  Adventure      5.9


In [66]:
print("=== By Title ===")
print(recommend_movies(title="Spider-Man "))

print("\n=== By Description ===")
print(recommend_movies(description="space adventure aliens future war"))

print("\n=== With Genre Filter ===")
print(recommend_movies(title="Inception", genre="Action"))

=== By Title ===
                                   title      genre  ratings
9177       Spider-Man: Homecoming - 2017     Action      7.4
8011                    Red Pearl - 2016    Romance      7.9
108481  An Englishman in New York - 2009  Biography      7.2
142049           The Spider-Man 2 - 2021     Action      6.3
110703           The Spider-Man 2 - 2021    Romance      6.3

=== By Description ===
                            title   genre  ratings
161160    Panda vs. Aliens - 2021   Scifi      3.5
163033    Panda vs. Aliens - 2021  Family      3.5
137833  Invaders from Mars - 1953  Horror      6.2
121848  Invaders from Mars - 1953   Scifi      6.2
8452            Earthbound - 1981   Scifi      4.4

=== With Genre Filter ===
                         title   genre  ratings
112694  Dost Garibon Ka - 1989  Action      4.7
94077     Do Boond Pani - 1971  Action      6.4
